# AutoPath – Thực nghiệm trên tập TP.HCM (Kaggle)

Notebook này **không huấn luyện lại**: dùng checkpoint đã huấn luyện ở notebook TuSimple (Version 1, 03/11/2026).

**Input cần gắn:**
1. Dataset `autopath-hcmc`: các tệp `autopath_hcmc_part_*.bin` (code + khung hình + đoạn video TP.HCM, cắt nhỏ để upload).
2. *Notebook Output* của notebook huấn luyện TuSimple (chứa `runs/*/best.pt` hoặc `autopath_outputs.zip`).
3. (Khi chạy lại) *Notebook Output* của chính notebook này (version trước) để khôi phục các bước đã xong.

*Settings → Accelerator → GPU T4 x2*, bật *Internet*, rồi **Save Version → Save & Run All**.

Mỗi bước ghi dấu vào `hcmc_out/_done/`; version sau tự bỏ qua các bước đã xong. Khi có nhãn: thêm `labels.json` vào dataset (New Version) — bước đánh giá tự chạy vì mã băm nhãn thay đổi.


In [ ]:
import os, glob, shutil, json, subprocess, zipfile
WORK = '/kaggle/working'
DATA = f'{WORK}/data'
def find_all(pattern, depth=7):
    out = []
    for d in range(0, depth):
        out += glob.glob('/kaggle/input/' + '*/' * d + pattern)
    return sorted(set(out))

# 1) Ghép các phần của dataset autopath-hcmc rồi giải nén
parts = find_all('autopath_hcmc_part_*.bin')
print('Số phần dataset:', len(parts))
if not os.path.exists(f'{DATA}/hcmc'):
    os.makedirs(DATA, exist_ok=True)
    with open(f'{WORK}/hcmc_data.zip', 'wb') as fo:
        for p in parts:
            fo.write(open(p, 'rb').read())
    zipfile.ZipFile(f'{WORK}/hcmc_data.zip').extractall(DATA)
    os.remove(f'{WORK}/hcmc_data.zip')
labels_in = find_all('labels.json')          # nhãn có thể được thêm riêng vào dataset
if labels_in:
    shutil.copy(labels_in[0], f'{DATA}/hcmc/labels.json')
HCMC = f'{DATA}/hcmc'
print('hcmc:', os.listdir(HCMC), '| labels.json:', os.path.exists(f'{HCMC}/labels.json'))

# 2) Checkpoint từ output notebook huấn luyện TuSimple
ck = [p for p in find_all('runs/res18_aug/best.pt') if '/hcmc_out/' not in p]
if ck:
    RUNS = os.path.dirname(os.path.dirname(ck[0]))   # .../runs
else:
    z = find_all('autopath_outputs.zip')
    assert z, 'Chưa gắn Notebook Output của notebook huấn luyện TuSimple'
    zipfile.ZipFile(z[0]).extractall(f'{WORK}/v1')
    RUNS = f'{WORK}/v1/runs'
print('runs:', RUNS, os.listdir(RUNS))
assert os.path.exists(f'{RUNS}/res18_aug/best.pt'), 'Không tìm thấy checkpoint res18_aug'

# 3) Code
os.chdir(f'{DATA}/AutoPath')
!pip -q install albumentations gdown ultralytics
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# Kết quả của version trước (nếu đã gắn Notebook Output) -> khôi phục để không chạy lại
PREV = [os.path.dirname(p) for p in find_all('hcmc_out/_done')]
print('Kết quả cũ:', PREV)
# AutoSteer 2.0: mã nguồn + trọng số (giữ trong /kaggle/working/weights để lần sau dùng lại)
if not os.path.exists('auto_steer'):
    !git clone -q https://github.com/autowarefoundation/auto_steer.git
os.makedirs(f'{WORK}/weights', exist_ok=True)
W = f'{WORK}/weights/autosteer.pth'
old = find_all('weights/autosteer.pth')
if not os.path.exists(W) and old:
    shutil.copy(old[0], W)
if not os.path.exists(W):
    !gdown -q https://drive.google.com/uc?id=1-wfPyu7HId7YDSh_T_DkB3Ma1zPc7Fu_ -O {W}
print('AutoSteer weights:', os.path.exists(W))


In [ ]:
OUT = f'{WORK}/hcmc_out'
for p in PREV:   # khôi phục kết quả version trước
    if os.path.abspath(p) != OUT:
        shutil.copytree(p, OUT, dirs_exist_ok=True)
# Hình tổng hợp phụ thuộc vào các bước khác: luôn vẽ lại
for m in glob.glob(f'{OUT}/_done/predict_figure*') + glob.glob(f'{OUT}/_done/figures_*'):
    os.remove(m)
!python tools/run_hcmc_pipeline.py --hcmc {HCMC} --runs {RUNS} --out {OUT} \
    --autosteer-repo auto_steer --autosteer-weights {W} --yolo yolov8n.pt


In [ ]:
# Xem nhanh kết quả
from IPython.display import Image, display
for f in ['hcmc_compare_methods.jpg', 'hcmc_predictions_res18.jpg', 'figures/ema_alpha_tradeoff.png']:
    p = f'{WORK}/hcmc_out/{f}'
    if os.path.exists(p):
        print(f); display(Image(p, width=900))
for p in ['alpha_hcmc_mean.csv', 'figures/tables.md', 'errors.log']:
    p = f'{WORK}/hcmc_out/{p}'
    if os.path.exists(p):
        print('=' * 20, p); print(open(p, encoding='utf-8').read())
for p in sorted(glob.glob(f'{WORK}/hcmc_out/demo/*.json')):
    print(os.path.basename(p), json.load(open(p)))


In [ ]:
# Đóng gói để tải về (Output -> hcmc_outputs.zip). Thư mục hcmc_out được giữ nguyên để version sau khôi phục.
!cd {WORK} && rm -f hcmc_outputs.zip && zip -qr hcmc_outputs.zip hcmc_out && ls -la hcmc_outputs.zip


## (Tùy chọn) Huấn luyện lại có điểm lưu
`train.py --resume` tiếp tục từ `runs/<tên>/last.pt` nếu Kaggle bị ngắt, và bỏ qua lần chạy đã có tệp `DONE`. Chỉ cần khi muốn huấn luyện thêm (cần gắn dataset TuSimple):
```
!cp -r {RUNS} {WORK}/runs   # chép checkpoint cũ sang thư mục ghi được
!python train.py --data-root <TuSimple> --backbone resnet18 --name res18_aug --out-dir {WORK}/runs --resume
```
